<a href="https://colab.research.google.com/github/gmauricio-toledo/math-ML/blob/main/Notebooks/13-NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Procesamiento de Lenguaje Natural</h1>

En esta notebook se presentan algunas técnicas y herramientas fundamentales del NLP para manejar documentos de texto en procesos de Machine Learning.

Particularmente veremos:

0. Stopwords
1. Modelos BOW y TF-IDF
2. Modelos basados en transformadores

# Stopwords

Los **stopwords** son palabras comunes en un idioma (como 'el', 'la', 'un', 'una', 'y', 'o', etc.) que generalmente se eliminan del texto antes del procesamiento de lenguaje natural. Esto se hace porque estas palabras no suelen aportar un significado sustancial al contenido del texto y pueden introducir ruido en los análisis, especialmente en tareas como la clasificación de texto o la recuperación de información. Al eliminarlas, se reduce la dimensionalidad de los datos y se mejora el rendimiento de muchos algoritmos de aprendizaje automático.

**NLTK** (Natural Language Toolkit) es una biblioteca de Python para el procesamiento de lenguaje natural. Proporciona herramientas para la tokenización, el stemming, la lematización, el etiquetado de partes del discurso, la clasificación, el análisis sintáctico y muchos otros aspectos del NLP.

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
spanish_stopwords = stopwords.words('spanish')
print(spanish_stopwords)

#Representaciones vectoriales de documentos

El hecho de usar `CountVectorizer` o el `TfidfVectorizer` para vectorizar los documentos se puede ver de dos maneras equivalentes:

1. **Extracción de features**: El texto no posee intrínsecamente features que definan al texto, por lo que estas técnicas transforman las palabras en características numéricas basadas en su frecuencia o importancia.
2. **Representación vectorial** de documentos: Estas herramientas convierten los textos en vectores numéricos, permitiendo su procesamiento matemático por medio de algoritmos de ML. A diferencia de los word embeddings, que capturan significado y contexto, estos métodos se centran en la representación superficial del texto.


<img src="https://drive.google.com/uc?id=1FxlQrTa2Vg7_QGRVlfYIM5z9tSmAzXZW" alt="alt text" width="500">



## Ejemplo 1

El corpus:

In [ ]:
documentos = [
        'buen hotel, economico y recomendable',
        'el niño juega en el parque',
        'buen automovil, recomendable'
        ]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizador = CountVectorizer()
X = vectorizador.fit_transform(documentos)
X = X.toarray()
print(X) # X es una matriz sparse, con toarray() la convertimos a numpy

In [ ]:
import pandas as pd

pd.DataFrame(X,
             columns=vectorizador.get_feature_names_out(),
             index=documentos
             )

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(documentos)
X = X.toarray()
print(X) # X es una matriz sparse, con toarray() la convertimos a numpy

In [ ]:
pd.DataFrame(X,
             columns=vectorizador.get_feature_names_out(),
             index=documentos
             )

### Sentence Transformers

Modelos disponibles: https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

In [ ]:
from sentence_transformers import SentenceTransformer

# Cargar modelo pre-entrenado (escoger uno)
modelo = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Transformar textos a representaciones vectoriales
embeddings = modelo.encode(documentos)
print(f"Shape: {embeddings.shape}")
print(embeddings[:,:5])

## Ejemplo 2

In [ ]:
!gdown 1mE4dVNVqS1Xt28xz7ngHZrtGM0WTSrQ0

In [ ]:
import pandas as pd

df = pd.read_csv("noticias_unison.csv")
df


In [ ]:
texts = df["texto"].tolist()
texts[:2]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizador = CountVectorizer(
                            max_features=2000, # Sólo se tomarán en cuenta esta cantidad de palabras (más frecuentes)
                            stop_words=spanish_stopwords # Quitamos stopwords
                               )
X = vectorizador.fit_transform(texts)
X = X.toarray()

print(X.shape)

Este es nuestro conjunto de datos, ¿qué forma tiene? ¿cuántas variables tiene?

Exploremos los vectores de documentos

In [ ]:
n_entradas = 20

X[0,:n_entradas]

Veamos cuántas entradas son 0:

In [ ]:
import numpy as np

num_zero_entries = np.where(X.flatten()==0)[0].shape[0]
zero_entries_ratio = num_zero_entries/(X.shape[0]*X.shape[1])
print(f"Porcentaje de entradas cero: {100*zero_entries_ratio:.2f}%")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

dim_red = PCA()
dim_red.fit(X)
varianza_acumulada = np.cumsum(dim_red.explained_variance_ratio_)

plt.figure()
plt.plot(varianza_acumulada)
plt.axhline(y=0.95, color='r', linestyle='--')
plt.show()

In [ ]:
X_pca = dim_red.transform(X)

plt.figure()
plt.scatter(X_pca[:,0],X_pca[:,1])
plt.show()

In [ ]:
import plotly.express as px

# Visualización 3D para X_pca
fig_X_pca = px.scatter_3d(
    X_pca,
    x=0, y=1, z=2, # Usar las primeras 3 componentes principales para 3D
    hover_name=[t[:150] for t in texts], # Mostrar el texto original al hacer hover
    color_discrete_sequence=['green'], # Un solo color
    title='Visualización 3D de X_pca (CountVectorizer)'
)

fig_X_pca.update_layout(scene=dict(
    xaxis=dict(showticklabels=False, showbackground=False, zeroline=False, visible=False),
    yaxis=dict(showticklabels=False, showbackground=False, zeroline=False, visible=False),
    zaxis=dict(showticklabels=False, showbackground=False, zeroline=False, visible=False)
))

fig_X_pca.write_html('X_pca_3d.html')

In [ ]:
from sentence_transformers import SentenceTransformer

# Cargar modelo pre-entrenado (escoger uno)
modelo = SentenceTransformer('all-MiniLM-L6-v2')

# Transformar textos a representaciones vectoriales
embeddings = modelo.encode(texts)
print(f"Shape: {embeddings.shape}")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

dim_red = PCA()
embeddings_pca = dim_red.fit_transform(embeddings)
varianza_acumulada = np.cumsum(dim_red.explained_variance_ratio_)

plt.figure()
plt.plot(varianza_acumulada)
plt.axhline(y=0.95, color='r', linestyle='--')
plt.show()

In [ ]:
plt.figure()
plt.scatter(embeddings_pca[:,0],embeddings_pca[:,1])
plt.show()

In [ ]:
import plotly.express as px

# Visualización 3D para embeddings_pca
fig_embeddings = px.scatter_3d(
    embeddings_pca,
    x=0, y=1, z=2, # Usar las primeras 3 componentes principales
    hover_name=[t[:150] for t in texts], # Mostrar el texto original al hacer hover
    color_discrete_sequence=['blue'],
    title='Visualización 3D de embeddings_pca'
)

fig_embeddings.update_layout(scene=dict(
    xaxis=dict(showticklabels=False, showbackground=False, zeroline=False, visible=False),
    yaxis=dict(showticklabels=False, showbackground=False, zeroline=False, visible=False),
    zaxis=dict(showticklabels=False, showbackground=False, zeroline=False, visible=False)
))

fig_embeddings.write_html('embeddings_pca_3d.html')